# MedCore Models — Training + Tests

This notebook contains the three trained models from the MedCore dashboard
(`train_risk_model`, `train_noshow_model`, `train_feedback_noshow_model`)
and testing of these models

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RNG_SEED = 7


def classification_metrics(y_true, proba, threshold=0.5):
    """Accuracy/Precision/Recall/F1 at a given probability threshold,
    plus AUC (threshold-independent). zero_division=0 so an
    all-one-class test fold reports 0 instead of raising a warning/error."""
    y_pred = (proba >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, proba),
    }

## 2. Model code 

In [2]:
VALID_RATINGS = {1, 2, 3, 4, 5}

RISK_NUM_FEATS = [
    "wait_days", "overbooking_index", "avg_visit_duration_min",
    "manual_workload_multiplier", "years_experience", "provider_panel_size",
    "annual_budget", "base_wait_days", "claim_denial_risk", "lab_delay_risk",
]
RISK_BOOL_FEATS = ["chronic_diabetes", "chronic_hypertension"]
RISK_CAT_FEATS = ["department_name", "specialty", "visit_type", "insurance_type", "region", "gender"]

RISK_LABELS = {
    "wait_days": "Longer wait time",
    "overbooking_index": "Higher overbooking at the clinic",
    "avg_visit_duration_min": "Longer average visit duration",
    "manual_workload_multiplier": "Higher manual staff workload",
    "years_experience": "More provider experience",
    "provider_panel_size": "Larger provider patient panel",
    "annual_budget": "Higher department budget",
    "base_wait_days": "Higher department baseline wait",
    "claim_denial_risk": "Higher claim-denial risk (dept.)",
    "lab_delay_risk": "Higher lab-delay risk (dept.)",
    "chronic_diabetes": "Patient has diabetes",
    "chronic_hypertension": "Patient has hypertension",
}
RISK_CAT_LABELS = {
    "department_name": "Department",
    "specialty": "Specialty",
    "visit_type": "Visit type",
    "insurance_type": "Insurance",
    "region": "Region",
    "gender": "Gender",
}


def friendly_feature_name(raw_name):
    """Turn an sklearn ColumnTransformer feature name (e.g.
    'num__wait_days' or 'cat__insurance_type_Self-Pay') into a short,
    plain-English label for charts."""
    if raw_name.startswith("num__"):
        base = raw_name[len("num__"):]
        return RISK_LABELS.get(base, base.replace("_", " ").capitalize())
    if raw_name.startswith("cat__"):
        rest = raw_name[len("cat__"):]
        for feat, label in RISK_CAT_LABELS.items():
            prefix = feat + "_"
            if rest.startswith(prefix):
                value = rest[len(prefix):]
                return f"{label}: {value}"
        return rest.replace("_", " ")
    return raw_name

### AI Model #1 — pre-visit predictive risk model

In [3]:
def train_risk_model(df):
    """Predicts P(patient will be at-risk / dissatisfied) using only
    information available once an appointment is booked — no rating,
    comment, or complaint_category is used as an input feature."""
    d = df.dropna(subset=RISK_NUM_FEATS + RISK_CAT_FEATS).copy()
    for c in RISK_BOOL_FEATS:
        d[c] = d[c].astype(int)

    X = d[RISK_NUM_FEATS + RISK_BOOL_FEATS + RISK_CAT_FEATS]
    y = d["at_risk"].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pre = ColumnTransformer([
        ("num", StandardScaler(), RISK_NUM_FEATS + RISK_BOOL_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), RISK_CAT_FEATS),
    ])
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    full_auc = roc_auc_score(y_test, proba)
    metrics = classification_metrics(y_test, proba)

    base = LogisticRegression(max_iter=1000)
    base.fit(X_train[["wait_days"]], y_train)
    base_auc = roc_auc_score(y_test, base.predict_proba(X_test[["wait_days"]])[:, 1])

    feat_names = pre.get_feature_names_out()
    coefs = pipe.named_steps["clf"].coef_[0]
    importance = (
        pd.DataFrame({"feature": [friendly_feature_name(f) for f in feat_names], "coef": coefs})
        .assign(abs_coef=lambda x: x["coef"].abs())
        .sort_values("abs_coef", ascending=False)
        .head(8)
    )

    dept_profile = d.groupby("department_name")[RISK_NUM_FEATS].mean()
    dept_mode = d.groupby("department_name")[RISK_CAT_FEATS[1:] + RISK_BOOL_FEATS].agg(lambda s: s.mode().iat[0])

    return pipe, full_auc, base_auc, importance, dept_profile, dept_mode, metrics

### AI Model #2 — no-show predictor (money-lost tool)

In [4]:
def train_noshow_model(df):
    """Predicts the chance a single, specific appointment gets missed,
    using only details known when it's booked."""
    d = df.dropna(subset=RISK_NUM_FEATS + RISK_CAT_FEATS + ["status"]).copy()
    for c in RISK_BOOL_FEATS:
        d[c] = d[c].astype(int)
    d["no_show"] = (d["status"] == "No Show").astype(int)

    feat_cols = RISK_NUM_FEATS + RISK_BOOL_FEATS + RISK_CAT_FEATS
    X = d[feat_cols]
    y = d["no_show"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pre = ColumnTransformer([
        ("num", StandardScaler(), RISK_NUM_FEATS + RISK_BOOL_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), RISK_CAT_FEATS),
    ])
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
    pipe.fit(X_train, y_train)

    # (Not returned by the original dashboard function, but useful in the
    # notebook so we can report Accuracy/Precision/Recall/F1/AUC for this
    # model too.)
    proba = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    metrics = classification_metrics(y_test, proba)

    return pipe, auc, metrics

### AI Model #3 — feedback-based next-appointment no-show risk

In [5]:
def train_feedback_noshow_model(df):
    """Given a rating a patient just left, plus their wait time and visit
    type on the visit they're rating, predicts the chance they no-show
    their *next* appointment. Deliberately uses `rating` as an input."""
    d = df[df["rating"].isin(VALID_RATINGS)].copy()
    d = d.dropna(subset=["future_no_show_flag"])
    d["future_no_show_flag"] = d["future_no_show_flag"].astype(int)
    d["wait_days"] = d["wait_days"].fillna(d["wait_days"].median())

    visit_dummies = pd.get_dummies(d["visit_type"], prefix="visit_type")
    d = pd.concat([d, visit_dummies], axis=1)

    feat_cols = ["rating", "wait_days"] + list(visit_dummies.columns)
    X = d[feat_cols]
    y = d["future_no_show_flag"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    metrics = classification_metrics(y_test, proba)

    dept_wait_typical = d.groupby("department_name")["wait_days"].median()
    dept_visit_typical = d.groupby("department_name")["visit_type"].agg(lambda s: s.mode().iat[0])

    return model, feat_cols, list(visit_dummies.columns), dept_wait_typical, dept_visit_typical, auc, metrics

## 3. Load the dataset

Loads `patient_experience_master.csv` the same way the dashboard's
`load_data()` does (date parsing, numeric coercion for `rating`/
`wait_days`). 

In [6]:
DATA_FILE = "patient_experience_master.csv"  

df = pd.read_csv(
    DATA_FILE,
    parse_dates=["feedback_date", "appointment_date", "scheduled_date", "dob", "registration_date"],
)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["wait_days"] = pd.to_numeric(df["wait_days"], errors="coerce")
df["feedback_month"] = df["feedback_date"].dt.to_period("M").dt.to_timestamp()

if "at_risk" not in df.columns:
    df["at_risk"] = df["rating"] <= 3

print(f"Loaded {len(df):,} rows from {DATA_FILE}")
df.head()

Loaded 289,889 rows from patient_experience_master.csv


,feedback_id,appointment_id,patient_id,department_id,rating,at_risk,complaint_category,comment,feedback_date,patient_satisfaction_after_visit,...,manual_workload_multiplier,claim_denial_risk,lab_delay_risk,total_claim_amount,total_patient_paid,total_insurance_paid,n_claims,any_claim_denied,has_claim,feedback_month
0,F000000001,A000052897,P0052751,D038,4,False,Wait Time,I waited much longer than expected.,2024-01-29,Satisfied,...,1.58,0.386,0.205,0.00,0.00,0.00,0.0,False,False,2024-01-01
1,F000000002,A000679409,P0071824,D007,3,True,Wait Time,The wait was too long.,2025-05-14,At Risk,...,1.19,0.208,0.156,0.00,0.00,0.00,0.0,False,False,2025-05-01
2,F000000003,A000724365,P0085277,D045,3,True,Communication,Instructions were unclear.,2025-11-11,At Risk,...,0.88,0.215,0.228,0.00,0.00,0.00,0.0,False,False,2025-11-01
3,F000000004,A000303615,P0029668,D039,3,True,Communication,Communication could be better.,2023-11-24,At Risk,...,2.46,0.073,0.200,371.52,30.64,340.88,1.0,False,True,2023-11-01
4,F000000005,A000213081,P0014845,D027,3,True,Wait Time,The wait was too long.,2024-02-11,At Risk,...,1.13,0.159,0.205,377.46,49.91,327.55,1.0,False,True,2024-02-01


## 4. Train all three models and report Accuracy / Precision / Recall / F1 / AUC

All four classification metrics are computed at the standard 0.5
probability threshold (`classification_metrics()` above). AUC is also
included since it's threshold-independent and is the metric the
dashboard itself reports.

In [7]:
risk_pipe, risk_full_auc, risk_base_auc, risk_importance, risk_dept_profile, risk_dept_mode, risk_metrics = train_risk_model(df)
noshow_pipe, noshow_auc, noshow_metrics = train_noshow_model(df)
(feedback_model, feedback_feat_cols, feedback_visit_cols,
 feedback_dept_wait, feedback_dept_visit, feedback_auc, feedback_metrics) = train_feedback_noshow_model(df)

metrics_table = pd.DataFrame({
    "Model 1 — Pre-visit risk": risk_metrics,
    "Model 2 — Pre-visit no-show": noshow_metrics,
    "Model 3 — Feedback-based no-show": feedback_metrics,
}).T[["accuracy", "precision", "recall", "f1", "auc"]]
metrics_table.columns = ["Accuracy", "Precision", "Recall", "F1-score", "AUC"]

print("=== Classification metrics (test set, 0.5 threshold) ===")
print(metrics_table.round(3).to_string())
print()
print(f"(Model 1 wait_days-only baseline AUC: {risk_base_auc:.3f}, for comparison against its full-model AUC of {risk_full_auc:.3f})")
print()
print("Top feature drivers (risk model):")
print(risk_importance[["feature", "coef"]].to_string(index=False))

=== Classification metrics (test set, 0.5 threshold) ===
                                  Accuracy  Precision  Recall  F1-score    AUC
Model 1 — Pre-visit risk             0.647      0.567   0.366     0.445  0.665
Model 2 — Pre-visit no-show          0.887      0.000   0.000     0.000  0.633
Model 3 — Feedback-based no-show     0.888      0.558   0.121     0.200  0.793

(Model 1 wait_days-only baseline AUC: 0.664, for comparison against its full-model AUC of 0.665)

Top feature drivers (risk model):
              feature      coef
     Longer wait time  0.614255
   Visit type: Urgent -0.110286
Visit type: Procedure -0.089433
        Gender: Other -0.087846
       Gender: Female -0.086419
         Gender: Male -0.079095
Visit type: Follow-up -0.076475
  Insurance: Employer -0.068142


Model 1 (Pre-visit risk) achieved moderate performance (Accuracy = 0.647, AUC = 0.665), indicating it can distinguish between higher- and lower-risk patients better than chance, but has limited predictive power. The most influential predictor was longer wait time, suggesting patients waiting longer before their appointment are more likely to be classified as high risk. However, adding additional demographic and visit-related variables provided almost no improvement over using wait time alone (AUC 0.665 vs. 0.664).

Model 2 (Pre-visit no-show) achieved high accuracy (0.887) but failed to identify any no-show cases (Precision = Recall = F1 = 0). This indicates the model predicted nearly all patients as attending, likely due to class imbalance, making accuracy a misleading performance metric despite a moderate AUC of 0.633.

Model 3 (Feedback-based no-show) achieved the highest discrimination ability (AUC = 0.793), suggesting patient feedback contains useful information for predicting no-shows. However, at the default 0.5 decision threshold, recall remained low (0.121), meaning many actual no-show cases were still missed. Lowering the classification threshold or addressing class imbalance could improve its practical performance.

Overall, the results suggest that patient feedback provides the strongest predictive signal, while wait time is the dominant pre-visit risk factor. Future improvements should focus on handling class imbalance and optimizing the decision threshold to increase sensitivity to patients at risk of missing appointments.